# 03c — HTTP, validation, and FastAPI — Practice

Read the lesson and run the example. Implement each TODO, then run its checks. A NotImplementedError is expected until you write your answer. Try for 20–30 minutes before opening a hint; open the solution only after recording an attempt. Copy this notebook into work/ to keep your own version.

**Prerequisites:** earlier lessons in the course order. **Environment:** Install requirements/core.txt.

[Stage guide](../stages/03.md) · [Course map](../PLAN.md) · [Project](../../projects/stage03/README.md)

**Notebook time:** 2–4 hours for an initial pass; project, reading, retrieval practice, and independent transfer use the rest of the stage budget.

Learning objectives:

- Model request and response contracts
- Test HTTP errors without a live server
- Separate authentication and authorization


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets" / "financials.csv").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the learning repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def expect_error(error_type, operation):
    try:
        operation()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")

print("Course root:", ROOT)


## Lesson

HTTP requests have a method, path, headers, and optional body. GET reads; POST creates or triggers work; PUT usually replaces; PATCH partially updates; DELETE removes. Status codes communicate outcomes: 2xx success, 4xx client/request failures, 5xx server failures. A JSON response still needs a schema.

FastAPI maps functions to routes and Pydantic validates input. Dependency injection supplies shared resources such as database sessions and the authenticated principal. Middleware can record request duration or attach a request ID. Authentication establishes identity; authorization decides what that identity may access. Derive tenant scope on the server from verified identity, never from a caller-supplied tenant field.

Use tests at the HTTP boundary to verify status codes, bodies, malformed inputs, and permissions. An in-memory list is useful for this first exercise but loses data on restart and is not safe shared storage for multiple workers.


## Worked example

Predict the output before running. Change one input and explain the result.


In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field
app = FastAPI()
@app.get("/health")
def health():
    return {"status": "ok"}
assert TestClient(app).get("/health").json() == {"status": "ok"}


## Exercise 1: Request schema

Define Record(BaseModel) with category length 1..50 and amount finite >=0.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
class Record(BaseModel):
    pass


In [ ]:
assert Record(category="Food", amount=2).amount == 2
from pydantic import ValidationError
expect_error(ValidationError, lambda: Record(category="", amount=-1))
print("Exercise 1: checks passed")


<details><summary>Hint — open after trying</summary>

Import BaseModel and Field before defining your class.

</details>


## Exercise 2: Create endpoint

Write make_app() with POST /records returning the validated record with status 201.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def make_app():
    raise NotImplementedError


In [ ]:
client = TestClient(make_app())
assert client.post("/records", json={"category": "Food", "amount": 2}).status_code == 201
assert client.post("/records", json={"category": "", "amount": -1}).status_code == 422
print("Exercise 2: checks passed")


<details><summary>Hint — open after trying</summary>

Annotate the body with your Pydantic model.

</details>


## Exercise 3: Pagination contract

Return a slice for offset>=0 and 1<=limit<=100; reject other bounds.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def paginate(rows, offset=0, limit=20):
    raise NotImplementedError


In [ ]:
assert paginate(list(range(10)), 3, 2) == [3, 4]
expect_error(ValueError, lambda: paginate([], 0, 1000))
print("Exercise 3: checks passed")


<details><summary>Hint — open after trying</summary>

Bound resource usage at the API boundary.

</details>


## Independent transfer

Build Stage 03’s persistent API with GET /records, /summary, /statistics and POST /records. Add auth, request IDs, tests, and PostgreSQL ingestion. Restart it and prove persistence.

Record your implementation, errors, measurements, and explanation in work/. See the project’s ANSWERS.md after attempting the brief.


## Explain without code

1. What does 422 mean here?

2. Why inject a database session?


## Reading and review

- [Primary reference 1](https://fastapi.tiangolo.com/tutorial/)

After 2, 7, and 30 days: explain the concept from memory, solve a changed-input version, and log any gaps in progress.md.
